# 05b - ResNet1D leve para sinais brutos

**Objetivo:** treinar uma CNN 1D residual leve usando sinais brutos do PTB-XL, preferencialmente `records500`.

Esta arquitetura foi adicionada porque os modelos clássicos com features estatísticas ficaram abaixo de 80% de acurácia. A rede usa o sinal temporal bruto do ECG, mantendo os mesmos rótulos, folds e métricas. O teste é usado apenas uma vez, após a seleção do melhor checkpoint por `val_loss` no fold 9.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tcc_ecg.config import load_config
from tcc_ecg.data import prepare_metadata
from tcc_ecg.deep_learning import get_resnet1d_config, train_resnet1d
from tcc_ecg.utils import setup_logging

setup_logging()
config = load_config()
resnet_config = get_resnet1d_config(config)
config['data']['signal_frequency'] = int(resnet_config['signal_frequency'])

print('Frequencia usada pela ResNet1D:', config['data']['signal_frequency'])
print('Epocas maximas:', resnet_config['epochs'])
print('Batch size:', resnet_config['batch_size'])

Frequencia usada pela ResNet1D: 500
Epocas maximas: 12
Batch size: 64


## Treinamento

A normalização por canal é calculada somente com os registros de treino. O checkpoint salvo em `models/resnet1d_best.pt` corresponde ao menor `val_loss` observado no fold 9.

In [2]:
metadata = prepare_metadata(config, save_summary=False)
results = train_resnet1d(metadata, config)
display(pd.DataFrame([results['metrics']]))
print('Checkpoint:', results['checkpoint'])
print('Device:', results['device'])

,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,model,split,signal_frequency,smote
0,0.750303,0.738553,0.662354,0.738553,0.669774,0.761886,resnet1d_light,test,500,False


Checkpoint: C:\dev\personal\ptb-xl\models\resnet1d_best.pt
Device: cpu
